# Ensemble Facility-Level PUE/WUE Results — Exploratory Plots

Exploratory plots over the 20-GCM SSP3-7.0 ensemble facility-point PUE/WUE
results in `outputs/ensemble_pue_wue_facilities/` (produced by
`scripts/run_pue_wue_facilities.py --backend fast`), sampled at the 109
IM3/PNNL Oregon data-center points. See `docs/tasks/ensemble_computation.md`
for the task this extends (full-grid coverage is still blocked -- see that
doc's status notes -- so this is deliberately facility-point, not gridded).

## Scope and what these numbers actually are
- **Ensemble**: 20 of LOCA2's 27 CMIP6 GCMs (the ones with SSP3-7.0 for both
  temperature and its humidity companion -- see
  `outputs/ensemble_pue_wue/gcm_manifest.csv`), one realization each.
- **Archetypes** (Lei & Masanet 2022's Cases 1/2/5, `cooling_archetypes.py`):
  `ae-chiller` (airside economizer + adiabatic cooling + water-cooled
  chiller -- the paper's own Prineville/The Dalles validation configuration),
  `we-chiller` (waterside economizer + water-cooled chiller), `chiller-only`
  (midsize, no economizer).
- **Backend**: `fast` (vectorized-numpy psychrometrics,
  `cooling_archetypes_fast.py`), not the CoolProp reference implementation --
  a follow-up accuracy check against real Oregon daily conditions (3 sites,
  6 years, all 3 archetypes/periods) found PUE agrees to well under 0.25%
  everywhere, WUE agrees closely except **Case 1 (`ae-chiller`) on hot summer
  days**, where its error runs ~10x the other two archetypes' (still small in
  absolute terms, ~0.5-0.8% of WUE's range) -- keep that in mind wherever
  `ae-chiller` WUE is read at face value below.
- **Periods**: historical (1985-2014), mid-century (2045-2074), end-of-century
  (2075-2100, 26 years not 30 -- see the methods doc's Time Horizons section).
- **Scenario**: SSP3-7.0 -- a different forcing pathway than Section 1.1's
  heat-risk work (SSP5-8.5), not yet reconciled in the paper.

This is facility-point sampling (109 real Oregon data-center locations), not
the full-grid deliverable the ensemble task was originally scoped for --
the full-grid CoolProp run turned out to be computationally infeasible
(~5 days/GCM single-threaded) and the fast-numpy path hasn't been wired into
the gridded script pending the accuracy check above.


In [ ]:
import re
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap

ROOT = Path('/home/server/pi/homes/queenl/projects/climate-risk-dc')
FAC_DIR = ROOT / 'outputs' / 'ensemble_pue_wue_facilities'
DC_CSV = ROOT / 'data' / 'data_centers_im3_pnnl' / 'im3_open_source_data_center_atlas_v2026.02.09.csv'


## Palette and chart style

Same validated reference palette as `facility_PUE_explore.ipynb` (the
single-GCM 1.2 notebook) -- period colors carry the same meaning across both
notebooks (blue=historical, orange=mid-century, aqua=end-of-century), so a
reader moving between the two doesn't have to relearn the color key. See the
`dataviz` skill's `references/palette.md` for the full validated set.

Archetype is handled as small multiples (one panel per archetype) rather
than a fourth categorical color, and per-GCM ensemble spread is shown as a
box + jittered points in a single muted ink rather than 20 distinct hues --
per the palette's own rule that an entity count this high folds into small
multiples or a distribution encoding, not a generated color per series.


In [ ]:
# Reference palette (light mode) -- see dataviz skill references/palette.md
INK_PRIMARY = '#0b0b0b'
INK_SECONDARY = '#52514e'
INK_MUTED = '#898781'
GRIDLINE = '#e1e0d9'
AXIS_LINE = '#c3c2b7'
SERIES_BLUE = '#2a78d6'      # categorical slot 1 -- historical period
SERIES_ORANGE = '#eb6834'    # categorical slot 2 -- mid-century period
SERIES_AQUA = '#1baf7a'      # categorical slot 3 -- end-of-century period

PERIOD_COLORS = {
    'historical': SERIES_BLUE,
    'midcentury': SERIES_ORANGE,
    'endcentury': SERIES_AQUA,
}
PERIOD_LABELS = {
    'historical': 'Historical (1985-2014)',
    'midcentury': 'Mid-century (2045-2074)',
    'endcentury': 'End-of-century (2075-2100)',
}
PERIOD_ORDER = ['historical', 'midcentury', 'endcentury']

ARCHETYPE_LABELS = {
    'ae-chiller': 'Case 1: airside economizer +\nadiabatic cooling',
    'we-chiller': 'Case 2: waterside economizer',
    'chiller-only': 'Case 5: chiller only\n(no economizer)',
}
ARCHETYPE_ORDER = ['ae-chiller', 'we-chiller', 'chiller-only']

# Sequential one-hue blue ramp, light (near-zero) -> dark (high magnitude)
SEQUENTIAL_BLUE_STEPS = [
    '#cde2fb', '#b7d3f6', '#9ec5f4', '#86b6ef', '#6da7ec', '#5598e7',
    '#3987e5', '#2a78d6', '#256abf', '#1c5cab', '#184f95', '#104281', '#0d366b',
]
SEQUENTIAL_BLUE_CMAP = LinearSegmentedColormap.from_list('sequential_blue', SEQUENTIAL_BLUE_STEPS)

plt.rcParams.update({
    'figure.dpi': 140,
    'axes.edgecolor': AXIS_LINE,
    'axes.labelcolor': INK_SECONDARY,
    'axes.titlecolor': INK_PRIMARY,
    'text.color': INK_PRIMARY,
    'xtick.color': INK_MUTED,
    'ytick.color': INK_MUTED,
    'grid.color': GRIDLINE,
    'axes.grid': True,
    'grid.linewidth': 0.8,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
})


def _ensemble_box(ax, values_by_period, color, jitter_seed=0):
    """Box (period color) + jittered per-GCM points (muted ink) at each period position."""
    rng = np.random.default_rng(jitter_seed)
    positions = np.arange(1, len(PERIOD_ORDER) + 1)
    data = [values_by_period.get(p, []) for p in PERIOD_ORDER]
    bp = ax.boxplot(
        data, positions=positions, widths=0.5, patch_artist=True, showfliers=False,
        medianprops={'color': INK_PRIMARY, 'linewidth': 1.5},
        whiskerprops={'color': AXIS_LINE, 'linewidth': 1.0},
        capprops={'color': AXIS_LINE, 'linewidth': 1.0},
        boxprops={'facecolor': color, 'edgecolor': AXIS_LINE, 'alpha': 0.35, 'linewidth': 1.0},
    )
    for pos, vals in zip(positions, data):
        if len(vals) == 0:
            continue
        x = pos + rng.uniform(-0.12, 0.12, size=len(vals))
        ax.scatter(x, vals, s=14, color=INK_MUTED, alpha=0.7, zorder=3, linewidth=0)
    ax.set_xticks(positions)
    ax.set_xticklabels([PERIOD_LABELS[p].split(' (')[0] for p in PERIOD_ORDER], fontsize=8.5)
    return bp


## Load and aggregate

Each of the 180 `facility_pue_wue_{gcm}_ssp370_{archetype}_{period}_fast.csv`
files holds ~1.19M daily rows (109 facilities x ~10,950 days). Loading all of
them raw would be ~214M rows -- instead each file is aggregated to one row
per facility (period-mean and 95th-percentile PUE/WUE) immediately after
loading, and the raw daily rows are discarded, so peak memory stays at one
file's worth (~1.19M rows) rather than the full ensemble.


In [ ]:
FNAME_RE = re.compile(
    r'facility_pue_wue_(?P<gcm>.+)_ssp370_(?P<archetype>ae-chiller|we-chiller|chiller-only)_'
    r'(?P<period>historical|midcentury|endcentury)_fast\.csv'
)

files = sorted(FAC_DIR.glob('facility_pue_wue_*_fast.csv'))
print(f'{len(files)} facility ensemble files found (expected 20 GCMs x 3 archetypes x 3 periods = 180)')

summary_parts = []
for f in files:
    m = FNAME_RE.match(f.name)
    if not m:
        raise ValueError(f'Unparseable filename: {f.name}')
    meta = m.groupdict()
    df = pd.read_csv(f, usecols=['facility_id', 'facility_name', 'pue_daily', 'wue_daily'])
    agg = df.groupby(['facility_id', 'facility_name'], as_index=False).agg(
        pue_mean=('pue_daily', 'mean'),
        pue_p95=('pue_daily', lambda s: s.quantile(0.95)),
        wue_mean=('wue_daily', 'mean'),
        wue_p95=('wue_daily', lambda s: s.quantile(0.95)),
    )
    agg['gcm'] = meta['gcm']
    agg['archetype'] = meta['archetype']
    agg['period'] = meta['period']
    summary_parts.append(agg)

summary = pd.concat(summary_parts, ignore_index=True)
print(f'{len(summary)} facility x gcm x archetype x period summary rows')
print(f'GCMs: {sorted(summary["gcm"].unique())}')

# Inline equivalent of climate_risk_dc.geo.load_oregon_data_centers, without
# taking a dependency on the package from this decoupled exploratory notebook
# (matches facility_PUE_explore.ipynb / facility_heat_indices_explore.ipynb,
# which read only the pre-computed CSV outputs, not the package itself).
dc_atlas = pd.read_csv(DC_CSV)
facilities_geo = dc_atlas[dc_atlas['state_abb'] == 'OR'][['id', 'lon', 'lat']].rename(columns={'id': 'facility_id'})
summary = summary.merge(facilities_geo, on='facility_id', how='left')
assert summary['lon'].notna().all(), 'Some facility_ids failed to join to the facility inventory'

# Plausibility check: paper's Fig. 6 AE-archetype contour puts typical PUE in
# roughly 1.0-1.2 for this cooling design -- see docs/tasks/1.2-pue-analysis.md
# Task 5. Not a strict pass/fail bar (generic Table B.1 parameters, not
# facility-calibrated), just a sanity anchor.
hist_ae = summary.query("archetype == 'ae-chiller' and period == 'historical'")['pue_mean']
print(f'Historical ae-chiller PUE range across ensemble: [{hist_ae.min():.3f}, {hist_ae.max():.3f}] '
      f'(paper\'s plausibility ballpark: ~1.0-1.2)')


## Ensemble spread of fleet-mean PUE, by period and archetype

Fleet mean (across all 109 facilities) per GCM, then the spread across the
20-member ensemble shown as a box (period color) with each GCM's own point
jittered on top in muted ink -- so the panel shows both the ensemble's
central tendency and which of it is inter-model spread versus a single
outlier model.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)
for ax, archetype in zip(axes, ARCHETYPE_ORDER):
    sub = fleet_mean[fleet_mean['archetype'] == archetype]
    values_by_period = {p: sub[sub['period'] == p]['pue_mean'].to_numpy() for p in PERIOD_ORDER}
    _ensemble_box(ax, values_by_period, SERIES_BLUE, jitter_seed=hash(archetype) % 1000)
    ax.set_title(ARCHETYPE_LABELS[archetype], fontsize=10)
axes[0].set_ylabel('Fleet-mean daily PUE')
fig.suptitle('Ensemble spread of fleet-mean PUE (20 GCMs, SSP3-7.0)', color=INK_PRIMARY, y=1.03)
fig.tight_layout()
plt.show()


## Same view for WUE

Water use effectiveness, same ensemble/period/archetype structure. Recall the
accuracy caveat above: `ae-chiller`'s WUE carries a larger (though still
small, ~0.5-0.8% of range) fast-vs-CoolProp approximation error on hot days
than the other two archetypes -- read its absolute WUE level with that in
mind, though the *relative* pattern across periods/GCMs should be unaffected
since the approximation error doesn't have an obvious warming-direction bias.


In [ ]:
fleet_mean_wue = summary.groupby(['gcm', 'archetype', 'period'], as_index=False)['wue_mean'].mean()

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5), sharey=True)
for ax, archetype in zip(axes, ARCHETYPE_ORDER):
    sub = fleet_mean_wue[fleet_mean_wue['archetype'] == archetype]
    values_by_period = {p: sub[sub['period'] == p]['wue_mean'].to_numpy() for p in PERIOD_ORDER}
    _ensemble_box(ax, values_by_period, SERIES_AQUA, jitter_seed=hash('wue' + archetype) % 1000)
    ax.set_title(ARCHETYPE_LABELS[archetype], fontsize=10)
axes[0].set_ylabel('Fleet-mean daily WUE (L/kWh)')
fig.suptitle('Ensemble spread of fleet-mean WUE (20 GCMs, SSP3-7.0)', color=INK_PRIMARY, y=1.03)
fig.tight_layout()
plt.show()


## Prineville and The Dalles: ensemble spot-check

These are the paper's own validation facilities for the `ae-chiller`
archetype (Meta Prineville, Google The Dalles) -- Task 5 of
`docs/tasks/1.2-pue-analysis.md` anchors plausibility at roughly PUE 1.0-1.2
for this cooling design (shaded band below), based on the paper's Fig. 6
AE-archetype contour; the paper's own facility-calibrated validation reported
-3.94%/-2.32% relative error at these two sites specifically, but those are
relative-error figures against non-public calibrated parameters, not an
absolute PUE target this generic-parameter version can be checked against
directly.


In [ ]:
SPOTCHECK_IDS = {19988712: 'Meta Prineville', 231769626: 'Google The Dalles'}

spotcheck = summary[
    (summary['facility_id'].isin(SPOTCHECK_IDS)) & (summary['archetype'] == 'ae-chiller')
].copy()
spotcheck['site'] = spotcheck['facility_id'].map(SPOTCHECK_IDS)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5), sharey=True)
for ax, site in zip(axes, SPOTCHECK_IDS.values()):
    sub = spotcheck[spotcheck['site'] == site]
    values_by_period = {p: sub[sub['period'] == p]['pue_mean'].to_numpy() for p in PERIOD_ORDER}
    ax.axhspan(1.0, 1.2, color=INK_MUTED, alpha=0.12, zorder=0)
    _ensemble_box(ax, values_by_period, SERIES_BLUE, jitter_seed=hash(site) % 1000)
    ax.set_title(site, fontsize=10)
axes[0].set_ylabel('Fleet-mean daily PUE (ae-chiller)')
fig.suptitle('Prineville / The Dalles ensemble spread vs. plausibility band', color=INK_PRIMARY, y=1.02)
fig.text(0.5, -0.02, 'Shaded band: paper\'s Fig. 6 AE-archetype plausibility range (~1.0-1.2), not a facility-specific target',
          ha='center', fontsize=8, color=INK_MUTED)
fig.tight_layout()
plt.show()


## Does cooling-system choice matter more than the warming signal?

End-of-century PUE/WUE across the three archetypes, ensemble spread shown
per archetype -- if the spread *between* archetypes (a design decision) is
larger than the spread *within* one archetype across the 20-GCM ensemble (a
climate-uncertainty band), that says which one dominates the projection.


In [ ]:
eoc = summary[summary['period'] == 'endcentury']
eoc_fleet = eoc.groupby(['gcm', 'archetype'], as_index=False).agg(pue_mean=('pue_mean', 'mean'), wue_mean=('wue_mean', 'mean'))

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
for ax, metric, label, color in [
    (axes[0], 'pue_mean', 'End-of-century fleet-mean PUE', SERIES_ORANGE),
    (axes[1], 'wue_mean', 'End-of-century fleet-mean WUE (L/kWh)', SERIES_AQUA),
]:
    rng = np.random.default_rng(0)
    positions = np.arange(1, len(ARCHETYPE_ORDER) + 1)
    data = [eoc_fleet[eoc_fleet['archetype'] == a][metric].to_numpy() for a in ARCHETYPE_ORDER]
    ax.boxplot(
        data, positions=positions, widths=0.5, patch_artist=True, showfliers=False,
        medianprops={'color': INK_PRIMARY, 'linewidth': 1.5},
        whiskerprops={'color': AXIS_LINE, 'linewidth': 1.0},
        capprops={'color': AXIS_LINE, 'linewidth': 1.0},
        boxprops={'facecolor': color, 'edgecolor': AXIS_LINE, 'alpha': 0.35, 'linewidth': 1.0},
    )
    for pos, vals in zip(positions, data):
        x = pos + rng.uniform(-0.12, 0.12, size=len(vals))
        ax.scatter(x, vals, s=14, color=INK_MUTED, alpha=0.7, zorder=3, linewidth=0)
    ax.set_xticks(positions)
    ax.set_xticklabels([ARCHETYPE_LABELS[a].split(':')[0] for a in ARCHETYPE_ORDER], fontsize=8.5)
    ax.set_title(label, fontsize=10)
fig.suptitle('Archetype vs. ensemble spread, end-of-century', color=INK_PRIMARY, y=1.03)
fig.tight_layout()
plt.show()


## Spatial pattern: projected PUE increase across Oregon

Per-facility ensemble-median PUE delta (end-of-century mean minus historical
mean), `ae-chiller` archetype -- ensemble median rather than any single GCM,
so this reflects the central estimate across all 20 models rather than one
model's idiosyncrasies.


In [ ]:
hist_pue = summary.query("archetype == 'ae-chiller' and period == 'historical'").groupby('facility_id')['pue_mean'].median()
eoc_pue = summary.query("archetype == 'ae-chiller' and period == 'endcentury'").groupby('facility_id')['pue_mean'].median()
delta = (eoc_pue - hist_pue).rename('pue_delta').reset_index()
delta = delta.merge(summary[['facility_id', 'lon', 'lat']].drop_duplicates('facility_id'), on='facility_id')

projection = ccrs.PlateCarree()
fig, ax = plt.subplots(figsize=(6.5, 7.5), subplot_kw={'projection': projection})
ax.set_extent([-125, -116, 42, 46.5], crs=projection)
ax.add_feature(cfeature.STATES.with_scale('50m'), edgecolor=AXIS_LINE, linewidth=0.7, zorder=1)
ax.add_feature(cfeature.BORDERS.with_scale('50m'), edgecolor=INK_SECONDARY, linewidth=0.8, zorder=1)
ax.add_feature(cfeature.COASTLINE.with_scale('50m'), edgecolor=AXIS_LINE, linewidth=0.8, zorder=1)

sc = ax.scatter(
    delta['lon'], delta['lat'], c=delta['pue_delta'], cmap=SEQUENTIAL_BLUE_CMAP,
    s=45, edgecolor='white', linewidth=0.4, zorder=2, transform=projection,
)
cbar = fig.colorbar(sc, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label('Ensemble-median PUE delta (end-of-century - historical)', color=INK_SECONDARY)
cbar.ax.tick_params(color=INK_MUTED, labelcolor=INK_MUTED)
ax.set_title('Projected PUE increase by facility (ae-chiller, ensemble median)', color=INK_PRIMARY, fontsize=11)
ax.grid(False)
fig.tight_layout()
plt.show()

print(f'Ensemble-median PUE delta range across facilities: [{delta["pue_delta"].min():.4f}, {delta["pue_delta"].max():.4f}]')


## Caveats carried from `docs/tasks/ensemble_computation.md`

- **Facility-point, not gridded.** These are the 109 real IM3/PNNL Oregon
  facility locations, not full spatial coverage -- the actual gridded
  deliverable is still blocked (CoolProp infeasible at ~9,100 cells; the
  fast-numpy path is validated but not yet wired into the gridded script).
- **`fast` backend.** Validated against CoolProp on real Oregon conditions
  (3 sites, 6 years spanning all 3 periods): PUE within ~0.25% everywhere;
  WUE within ~10x tighter bounds for `we-chiller`/`chiller-only` than for
  `ae-chiller`, whose hot-day WUE error is the one place real (if still
  small) divergence shows up.
- **SSP3-7.0**, not Section 1.1's SSP5-8.5 -- a different forcing pathway
  for cooling-efficiency risk than for heat risk within the same study.
- **One ensemble member per model** -- structural (inter-model) uncertainty
  only, not a given model's internal variability.
- **End-of-century window is 26 years** (2075-2100), not the full 30-year
  window used for the other two periods.
- Elevation-adjusted pressure (new relative to the single-GCM 1.2 pipeline)
  is applied here -- the two pipelines' PUE/WUE are not directly comparable
  on that basis alone.
